# AAI-500 Final Project — Heart Disease Classification
**Dataset:** UCI Heart Disease (Cleveland) · 303 records · 14 attributes  
**Target:** Binary — 0 = no disease, 1 = disease present  
**GitHub:** https://github.com/pdhinaka/AAI500-Final-Project

---

## How to use this notebook

This notebook is divided into four parts. Each team member owns one part. **Run all cells from top to bottom in order** — later sections depend on variables created earlier.

| Section | Owner | What to do |
|---------|-------|------------|
| 0. Setup | Everyone | Run once. Imports, data load, cleaning. Do not modify. |
| 1. EDA | **Mina** | Explore the data. Visualize distributions and relationships with the target. |
| 2. Modeling | **Pranav** | Build and tune the logistic regression model. |
| 3. Evaluation | **Sristi** | Evaluate model performance with metrics, ROC curve, and confusion matrix. |

**Before you start your section:** read the markdown cell at the top of your section carefully.  
**After each code cell:** add a short markdown cell with your key observation (2–4 sentences). These become the report.  
**AI use:** if you used Claude or another AI tool, add a comment at the top of that cell: `# Used Claude MM/DD/YY to help with [task]`

---

## Section 0 — Setup
**Owner: Everyone — run this first, do not modify.**

Loads all libraries, pulls the raw dataset from UCI, cleans it, and defines the shared variables `heart_clean`, `continuous_cols`, and `categorical_cols` that every section below depends on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import logit
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, accuracy_score, ConfusionMatrixDisplay
)

np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded.")

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

heart = pd.read_csv(URL, header=None, names=columns, na_values="?")

# Collapse 0-4 severity target to binary (standard practice in the literature)
heart["target"] = (heart["num"] > 0).astype(int)

print(f"Raw shape: {heart.shape}")
print(f"Missing values:\n{heart.isna().sum()[heart.isna().sum() > 0]}")
print(f"\nClass balance:\n{heart['target'].value_counts()}")

In [ ]:
# Drop 6 rows with missing values in ca and thal (~2% of data — acceptable to drop)
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
continuous_cols  = ["age", "trestbps", "chol", "thalach", "oldpeak"]

heart_clean = heart.dropna().copy()
heart_clean[categorical_cols] = heart_clean[categorical_cols].astype("category")

print(f"Clean shape: {heart_clean.shape}  ({len(heart) - len(heart_clean)} rows dropped)")
print(heart_clean.dtypes)

---
## Section 1 — Exploratory Data Analysis
**Owner: Mina**

Your job is to explore the data and tell the story of what's in it before any modeling happens. This section becomes the **EDA section of the technical report**.

You are working with `heart_clean`, `continuous_cols`, and `categorical_cols` — all defined in Section 0. Do not reload the data.

Write the code for each cell yourself. The comments tell you exactly what to produce. After each code cell, fill in the markdown observation cell below it.

- **1.1** Summary statistics for continuous predictors
- **1.2** Target class distribution — bar chart
- **1.3** Distributions of continuous predictors split by target — KDE or histogram
- **1.4** Boxplots of continuous predictors vs. target, with Welch's t-test results
- **1.5** Stacked bar charts of categorical predictors vs. target, with chi-square test results
- **1.6** Correlation heatmap of continuous predictors
- **1.7** Written summary: which 5–6 features look most predictive going into modeling?

---

In [ ]:
# 1.1 Summary statistics — continuous predictors
# Call .describe() on heart_clean using continuous_cols + ["target"].
# Look at mean, std, min, max. Flag anything unusual (e.g. suspiciously large max values).


*(Mina — 2–4 sentence observation. Lead with the result, not "this table shows...")*

In [ ]:
# 1.2 Target class distribution
# Create a bar chart showing the count and percentage of patients in each class (0 and 1).
# Label each bar with its count and percentage.


*(Mina — 2–4 sentence observation. Is the dataset balanced? Why does that matter for modeling?)*

In [ ]:
# 1.3 Distributions of continuous predictors split by target
# For each variable in continuous_cols, plot overlapping KDE curves for class 0 and class 1.
# Use a 1x5 subplot grid. Add a shared legend.
# Hint: loop over continuous_cols and use .plot.kde() or sns.kdeplot().


*(Mina — which variables show the clearest visual separation between classes?)*

In [ ]:
# 1.4 Boxplots: continuous predictors vs. target + Welch's t-test
# For each variable in continuous_cols, plot side-by-side boxplots (class 0 vs. class 1).
# Run a Welch's t-test (stats.ttest_ind with equal_var=False) for each variable.
# Collect results into a DataFrame with columns: variable, mean_no_disease, mean_disease, t, p_value.
# Display the DataFrame sorted by p_value.


*(Mina — which variables are statistically significant? Which are not? Any surprises?)*

In [ ]:
# 1.5 Categorical predictors vs. target — stacked bar charts + chi-square test
# For each variable in categorical_cols, create a stacked bar chart showing
# the proportion of class 0 and class 1 within each category level.
# Run a chi-square test of independence (stats.chi2_contingency) for each variable.
# Use pd.crosstab to build the contingency table.
# Collect results into a DataFrame with columns: variable, chi2, dof, p_value.
# Display sorted by p_value.


*(Mina — which categorical predictors are most strongly associated with the target? Any that show no association?)*

In [ ]:
# 1.6 Correlation heatmap — continuous predictors
# Compute the correlation matrix for continuous_cols using .corr().
# Plot as a heatmap using sns.heatmap with annot=True.
# Look for any notably high correlations between predictors (multicollinearity).


*(Mina — any variables strongly correlated with each other? Does anything concern you going into modeling?)*

### 1.7 EDA Summary
*(Mina — write 1 paragraph summarizing which 5–6 features look most predictive going into modeling, and why. This goes directly into the technical report. 4–6 sentences, lead with your top findings.)*

---
## Section 2 — Modeling
**Owner: Pranav**

Build the logistic regression model. This section becomes the **Model Selection section of the technical report**.

You are working with `heart_clean`, `continuous_cols`, and `categorical_cols` from Section 0. Do not reload the data.

**You must create these exact variable names — Sristi's section depends on them:**

| Variable | What it is |
|----------|------------|
| `X_test_scaled` | Scaled test features |
| `y_test` | True labels for the test set |
| `y_pred` | Predicted class labels (0 or 1) |
| `y_prob` | Predicted probabilities of class 1 |
| `model` | Fitted LogisticRegression object |

- **2.1** Feature selection — choose which predictors to include; justify any you drop
- **2.2** Dummy-encode categorical predictors using `pd.get_dummies` with `drop_first=True`
- **2.3** Train/test split — 80/20, `random_state=42`, `stratify=y`
- **2.4** Scale continuous features — fit `StandardScaler` on train only, apply to both
- **2.5** Fit `LogisticRegression` and generate `y_pred` and `y_prob` on the test set
- **2.6** Coefficient table — feature, coefficient, and exponentiated odds ratio
- **2.7** Written summary: model choice, features used, top predictors

---

In [ ]:
# 2.1 Feature selection
# Decide which columns to include as predictors based on the EDA findings.
# Create a list called `features` and subset heart_clean to those columns + target.
# If you drop any predictors (e.g. chol, fbs), note the reason in the markdown cell below.


*(Pranav — which features did you include? Did you drop any, and why?)*

In [ ]:
# 2.2 Dummy encode categorical predictors
# Use pd.get_dummies() with drop_first=True to convert categoricals to binary columns.
# Print the resulting shape and column list.


*(Pranav — how many columns after encoding? Does the number make sense given the category levels?)*

In [ ]:
# 2.3 Train/test split — 80/20
# Separate X and y, then call train_test_split with test_size=0.2, random_state=42, stratify=y.
# Print the size of each split and the class balance in both.


*(Pranav — does the class balance look consistent between train and test?)*

In [ ]:
# 2.4 Scale continuous features
# Create a StandardScaler. Fit it on X_train's continuous columns only.
# Apply the fitted scaler to both X_train and X_test (do not refit on test).
# Name the results X_train_scaled and X_test_scaled.
# Verify by printing .describe() on the scaled continuous columns — mean should be ~0, std ~1.


*(Pranav — confirm the scaling looks correct. Why do we fit only on train and not on test?)*

In [ ]:
# 2.5 Fit logistic regression
# Instantiate LogisticRegression(C=1.0, max_iter=1000, random_state=42).
# Fit on X_train_scaled and y_train.
# Generate y_pred = model.predict(X_test_scaled)
# Generate y_prob = model.predict_proba(X_test_scaled)[:, 1]
# Print training accuracy and test accuracy.


*(Pranav — how close are training and test accuracy? A large gap suggests overfitting.)*

In [ ]:
# 2.6 Coefficient table
# Build a DataFrame with three columns: feature, coefficient (model.coef_[0]), odds_ratio (np.exp of coefficient).
# Sort by absolute value of coefficient, descending.
# Display the top results — these are the features driving predictions most.


*(Pranav — which features have the largest positive and negative coefficients? Do these align with the EDA findings?)*

### 2.7 Modeling Summary
*(Pranav — write 1 paragraph for the technical report: model type and why it was chosen, features used, how it was trained, and which predictors carry the most weight. 4–6 sentences.)*

---
## Section 3 — Model Evaluation
**Owner: Sristi**

Assess how well the model performs. This section becomes the **Model Analysis section of the technical report**. Do not retrain the model — use the outputs Pranav created.

**Variables available from Section 2:**
- `X_test_scaled` — scaled test features
- `y_test` — true labels
- `y_pred` — predicted class labels
- `y_prob` — predicted probabilities of class 1
- `model` — the fitted LogisticRegression object

**Key terms to know:**
- **Sensitivity (recall):** of all patients who have disease, what fraction did the model catch?
- **Specificity:** of all patients who are healthy, what fraction did the model correctly clear?
- **AUC:** overall ability to rank diseased patients above healthy ones — 0.5 is random, 1.0 is perfect
- In a clinical setting, false negatives (missing a disease case) are more costly than false positives. Flag this in your write-up.

- **3.1** Confusion matrix — visualized and with raw counts
- **3.2** Classification report — precision, recall, F1 for both classes
- **3.3** ROC curve and AUC score
- **3.4** Sensitivity and specificity computed from the confusion matrix
- **3.5** Written summary: model performance and limitations

---

In [ ]:
# 3.1 Confusion matrix
# Compute cm = confusion_matrix(y_test, y_pred).
# Plot it using ConfusionMatrixDisplay with display_labels=["No disease", "Disease"].
# Also print the raw matrix and unpack: tn, fp, fn, tp = cm.ravel()
# You will need tn, fp, fn, tp in cell 3.4.


*(Sristi — note the false negative count. What does it mean clinically if the model misses a disease case?)*

In [ ]:
# 3.2 Classification report
# Call classification_report(y_test, y_pred, target_names=["No disease", "Disease"]).
# Focus on the Disease class recall — that is the sensitivity.


*(Sristi — what are the precision and recall for the Disease class? Which matters more in this context and why?)*

In [ ]:
# 3.3 ROC curve and AUC
# Compute fpr, tpr, thresholds = roc_curve(y_test, y_prob)
# Compute auc_score = roc_auc_score(y_test, y_prob)
# Plot the ROC curve with the diagonal random-classifier baseline.
# Label the curve with the AUC value in the legend.


*(Sristi — interpret the AUC. What does the shape of the curve tell you about the model?)*

In [ ]:
# 3.4 Sensitivity and specificity
# Using tn, fp, fn, tp from cell 3.1, compute:
#   sensitivity = tp / (tp + fn)
#   specificity = tn / (tn + fp)
#   accuracy    = (tp + tn) / (tp + tn + fp + fn)
# Print all four metrics: accuracy, sensitivity, specificity, AUC.


*(Sristi — discuss the sensitivity/specificity tradeoff. Is the current balance appropriate for a clinical screening tool?)*

### 3.5 Evaluation Summary and Limitations
*(Sristi — write 1 paragraph for the technical report. State accuracy, AUC, sensitivity, and specificity. Discuss at least 2 limitations — e.g. dataset size (297 records after cleaning), the clinical cost of false negatives, or generalizability concerns. 5–7 sentences.)*

---
## Section 4 — Conclusion
**Owner: Pranav (write after all three sections are complete)**

*(Pull together the key findings from EDA, modeling, and evaluation into one paragraph. What did the analysis show? What are the most important predictors? How well does the model perform? What would you do differently with more data or time? This becomes the Conclusion and Recommendations section of the report. 5–7 sentences.)*